# Notebook 05 — Difference Equations and Chaos

**Module:** 02 — Mathematics for Biology  
**Notebook:** 05 of 12  
**Prerequisites:** NB04  
**Time estimate:** 75 minutes

---
## Step 1 — Motivation

Not all biological processes are continuous. Salmon spawn annually. Insects breed
in discrete generations. A genomic sequencer produces discrete reads. For these,
a *difference equation* — a map from one generation to the next — is more natural
than an ODE. The logistic map $N_{t+1} = rN_t(1 - N_t)$, a one-line difference
equation, was shown by Robert May in 1976 to produce chaos — unpredictable,
deterministic dynamics that shattered the assumption that simple models produce
simple behavior.

---
## Step 2 — Intuition

An ODE gives the *rate of change*: $dN/dt = f(N)$.
A difference equation gives the *next value*: $N_{t+1} = g(N_t)$.

They are related — the Euler method (NB07) converts an ODE into a difference
equation with step size $h$ — but discrete maps can behave very differently from
their continuous analogues. Increase $r$ past 3.57 in the logistic map and the
population trajectory becomes chaotic: sensitive to initial conditions, apparently
random, but completely deterministic.

---
## Step 3 — Biological Background

**May (1976) — the logistic map in population biology:**

Robert May's *Nature* paper showed that the logistic difference equation:
$$N_{t+1} = r N_t (1 - N_t)$$
(where $N_t \in [0,1]$ is the normalised population) produces:
- $r < 1$: population goes extinct
- $1 < r < 3$: converges to a stable fixed point
- $3 < r < 3.57$: period-doubling bifurcations
- $r > 3.57$: chaos

The paper is considered foundational because it demonstrated that complex ecological
dynamics need not arise from complex models — deterministic chaos arises from a
single-parameter equation.

**Discrete vs. continuous models in practice:**
- Annual breeding cycles → discrete (insects, salmon, many plants)
- Continuous reproduction → ODEs (bacteria, mammalian cells in culture)
- Sequencing data → discrete reads per position

---
## Step 4 — Mathematical Explanation

**Logistic map:**
$$N_{t+1} = r N_t (1 - N_t), \quad N_t \in [0, 1], \quad r \in [0, 4]$$

**Fixed points** (where $N^* = r N^* (1 - N^*)$):
- $N^* = 0$ (always exists)
- $N^* = 1 - 1/r$ (exists for $r > 1$)

**Stability** — a fixed point $N^*$ is stable if $|g'(N^*)| < 1$, where $g(N) = rN(1-N)$:
$$g'(N) = r(1 - 2N)$$
$$g'(N^*) = r(1 - 2(1 - 1/r)) = 2 - r$$

So $N^* = 1 - 1/r$ is stable when $|2 - r| < 1$, i.e., $1 < r < 3$.
At $r = 3$, the fixed point becomes unstable and a period-2 orbit bifurcates.

**Sensitivity to initial conditions (chaos):**
Two trajectories starting at $N_0$ and $N_0 + \epsilon$ diverge exponentially.
Quantified by the Lyapunov exponent:
$$\lambda = \lim_{T\to\infty} \frac{1}{T} \sum_{t=0}^{T-1} \ln|g'(N_t)|$$
Chaos ↔ $\lambda > 0$.

---
## Step 5 — Computational Explanation

**Bifurcation diagram algorithm:**
1. For each $r$ value in `[rmin, rmax]`:
   a. Initialise $N_0 = 0.5$
   b. Iterate the map $n_{\text{burn}}$ times (transient — let it settle)
   c. Record the next $n_{\text{plot}}$ iterations
   d. Plot $(r, N_t)$ as a scatter point
2. Repeating for many $r$ values reveals the bifurcation structure.

---
## Step 6 — Python Implementation

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
# Cell 6.1 — Logistic map iteration
def logistic_map(r: float, N0: float, n_steps: int) -> np.ndarray:
    """
    Iterate the logistic map N_{t+1} = r * N_t * (1 - N_t).

    Parameters
    ----------
    r : float — growth rate parameter in [0, 4]
    N0 : float — initial population in [0, 1]
    n_steps : int — number of iterations

    Returns
    -------
    np.ndarray of shape (n_steps + 1,) — trajectory including N0
    """
    traj = np.empty(n_steps + 1)
    traj[0] = N0
    for t in range(n_steps):
        traj[t+1] = r * traj[t] * (1 - traj[t])
    return traj

# Show different dynamical regimes
r_cases = [(0.8, "Extinction"), (2.5, "Stable fixed point"), (3.2, "Period-2 orbit"), (3.9, "Chaos")]
for r, label in r_cases:
    traj = logistic_map(r, 0.5, 100)
    print(f"r={r}  ({label}): last 5 values = {traj[-5:].round(4)}")

In [ ]:
# Cell 6.2 — Bifurcation diagram
def bifurcation_diagram(r_min: float = 2.5, r_max: float = 4.0,
                         n_r: int = 800, n_burn: int = 300, n_plot: int = 100):
    """
    Compute points for the logistic map bifurcation diagram.

    Returns
    -------
    r_pts, N_pts : arrays of plotted (r, N) pairs
    """
    r_vals = np.linspace(r_min, r_max, n_r)
    r_pts, N_pts = [], []
    for r in r_vals:
        N = 0.5
        for _ in range(n_burn):
            N = r * N * (1 - N)
        for _ in range(n_plot):
            N = r * N * (1 - N)
            r_pts.append(r)
            N_pts.append(N)
    return np.array(r_pts), np.array(N_pts)

r_pts, N_pts = bifurcation_diagram()
print(f"Bifurcation data: {len(r_pts):,} points")

In [ ]:
# Cell 6.3 — Sensitivity to initial conditions (hallmark of chaos)
r_chaos = 3.9
N0_a = 0.500000
N0_b = 0.500001  # differ by 1e-6

traj_a = logistic_map(r_chaos, N0_a, 50)
traj_b = logistic_map(r_chaos, N0_b, 50)
divergence = np.abs(traj_a - traj_b)

# When do they become meaningfully different?
threshold = 0.1
t_diverge = np.argmax(divergence > threshold)
print(f"Initial difference: {N0_b - N0_a:.2e}")
print(f"Trajectories differ by > {threshold} at t = {t_diverge}")
print(f"\nLast 10 values of each trajectory:")
print(f"  A: {traj_a[-10:].round(3)}")
print(f"  B: {traj_b[-10:].round(3)}")

---
## Step 7 — Visualization

In [ ]:
# Cell 7.1 — Three-panel: regimes, bifurcation diagram, chaos
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Panel 1: four dynamical regimes
ax = axes[0]
colors = ["steelblue", "green", "orange", "tomato"]
for (r, label), color in zip(r_cases, colors):
    traj = logistic_map(r, 0.5, 50)
    ax.plot(traj, lw=1.5, color=color, label=f"r={r}")
ax.set_xlabel("Generation (t)"); ax.set_ylabel("N_t")
ax.set_title("Logistic map dynamical regimes")
ax.legend(fontsize=8)

# Panel 2: bifurcation diagram
ax = axes[1]
ax.scatter(r_pts, N_pts, s=0.2, c="black", alpha=0.3)
ax.set_xlabel("r"); ax.set_ylabel("N (after transient)")
ax.set_title("Bifurcation diagram")
ax.axvline(3.0, color="steelblue", ls="--", alpha=0.5, label="Period-2 onset (r=3)")
ax.legend(fontsize=8)

# Panel 3: sensitive dependence on initial conditions
ax = axes[2]
ax.semilogy(divergence, color="tomato", lw=1.5)
ax.axvline(t_diverge, color="gray", ls="--", alpha=0.7,
           label=f"t={t_diverge}: |ΔN|>{threshold}")
ax.set_xlabel("t"); ax.set_ylabel("|N_a(t) - N_b(t)|  (log)")
ax.set_title("Sensitive dependence (r=3.9)")
ax.legend()

plt.tight_layout()
plt.show()

---
## Step 8 — Exercises

1. Find the fixed points of the logistic map algebraically: set $N^* = r N^* (1 - N^*)$
   and solve. Verify your answers for $r = 2.5$ by running 500 iterations from $N_0 = 0.5$.
2. At $r = 3.2$ the map has a period-2 orbit. Find the two values $N_a, N_b$ such that
   $g(g(N_a)) = N_a$ (a period-2 cycle). Verify numerically.
3. Compute the Lyapunov exponent for $r = 2.5$ and $r = 3.9$ using the formula
   $\lambda = (1/T)\sum_{t=0}^{T-1} \ln|r(1 - 2N_t)|$. What is the sign in each case?
4. The continuous logistic ODE (NB04) never exhibits chaos. Write two sentences
   explaining why discrete maps can but continuous first-order ODEs cannot.

---
## Quiz — Active Recall

1. Write the logistic map equation from memory. What is the range of $r$ for chaos?
2. What is a fixed point? What makes one stable?
3. What is sensitive dependence on initial conditions? Give a biological example.
4. What did May (1976) demonstrate that surprised ecologists?
5. How does a bifurcation diagram show period-doubling?

---
## Papers Referenced

- May, R.M. (1976). Simple mathematical models with very complicated dynamics.
  *Nature*, 261, 459–467.

---
## Reflection

**Date completed:** ____________________

> *[Could you reproduce the bifurcation diagram from scratch? What surprised you about chaos arising from a one-line equation?]*

---
*Next: `06_first_order_odes_separable.ipynb`*